# Particles in NWES effect of derivatives of flowfield
We use finite differences to calculate the material derivative. Where because of the way parcels works we use an forward and backward derivative in time, at the moment we find spiky behavior in the slip velocity everytime we step to a new time slice which is not what we want we can try to solve this in 2 ways:
1) use derivative fieldsets 
2) load field into memory and not use the forward and backward derivative in time (this should solve it for sure)
method 2 should work for sure, method 1 not sure as their might be still some jump. 

In [ ]:
# import needed packages

#update reading in packages when rerunning this cell
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/nethome/4291387/Maxey_Riley_advection/Maxey_Riley_advection/src")
import numpy as np
import xarray as xr 
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from mpl_toolkits.axes_grid1 import Divider, Size
import matplotlib.cm as cm
from matplotlib import colormaps
import cartopy.crs as ccrs #for plotting on map
import cartopy as cart
from datetime import datetime, timedelta
from importlib import reload

import analysis_functions_xr as afxr
import analysis_functions as af
import particle_characteristics_functions as pcf

# plotstyle: 
plt.style.use('../python_style_Meike.mplstyle')
# markers
color_array = np.array(['c','orange','purple','green','navy'])
marker_array = np.array(['s','o','>','p'])
line_array = np.array(['-','--','-.',':'])
markerline_array = np.array(['-s','--o','-.>',':p'])

Magma = colormaps['magma']
Magmalist = Magma(np.linspace(0.2, 0.8, 3))#Magma(np.linspace(0.2, 0.9, 5))

In [ ]:
# set needed constants
Rearth = 6371 * 10**3 # in m,
deg2rad = np.pi / 180.
sec_in_hours= 3600


# properties drifters
B = 0.68
tau = 3196#2994.76 #2759.97
nparticles = 52511
chunck_time = 100
diameter = 0.25

#properties water
av_temp_NWES = 11.983276
av_salinity_NWES = 34.70449
rho_water = 1027 # kg/m3 
dynamic_viscosity_water = pcf.dynamic_viscosity_Sharqawy(av_temp_NWES,av_salinity_NWES/1000)
kinematic_viscosity_water = dynamic_viscosity_water / rho_water

In [ ]:
# import file
base_directory = '/storage/shared/oceanparcels/output_data/data_Meike/MR_advection/NWES/'
filename_original = 'inertial_SM_drag_Rep/NWES_start2023_09_01_end2023_09_03_RK4_B0680_tau3196_anti_beaching_cor_True_gradient_True_freq5min.zarr'

filename_derivate = 'inertial_SM_drag_Rep/NWES_start2023_09_01_end2023_09_03_RK4_B0680_tau3196_anti_beaching_cor_True_gradient_True_derivatives_field.zarr'

filename_loaded_time = 'inertial_drag_Rep/NWES_start2023_09_01_end2023_09_03_RK4_B0680_tau3196_anti_beaching_cor_True_gradient_True_loaded_time.zarr'

file = base_directory + filename_original
ds = xr.open_dataset(file)
# file = base_directory + filename_derivate
ds = xr.open_dataset(file,
                                            engine='zarr',
                                            chunks={'trajectory':nparticles, 'obs':chunck_time},
                                            drop_variables=['z'],
                                            decode_times=False)
Uslip = np.sqrt(ds.uslip**2 + ds.vslip**2) 
ds = ds.assign(Uslip = Uslip)
Rep_measured = pcf.Re_particle(Uslip,diameter, kinematic_viscosity_water)
ds = ds.assign(Rep_measured = Rep_measured)
ds.Uslip[:,0]=np.nan

In [ ]:
idcoast = 25#6432

fig = plt.figure()
h = [Size.Fixed(1.0), Size.Fixed(8)]
v = [Size.Fixed(0.7), Size.Fixed(6)]
divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)
ax = fig.add_axes(divider.get_position(),
                  axes_locator=divider.new_locator(nx=1, ny=1),projection=ccrs.PlateCarree())
# ax.plot(ds.isel(trajectory=idcoast).time/60,ds.isel(trajectory=idcoast).Rep_measured,'--o')
ax.plot(ds.isel(trajectory=idcoast).lon,ds.isel(trajectory=idcoast).lat)
ax.add_feature(cart.feature.LAND, facecolor='lightgrey')
ax.coastlines()

In [ ]:
fig, ax = plt.subplots()
idcoast = 25#6432
ax.plot(ds.isel(trajectory=idcoast).time[2::]/60,ds.isel(trajectory=idcoast).Rep_measured[2::],'--o')